In [29]:
import kagglehub
import pandas as pd
import os
import ast
import html
import re
import string
import unicodedata
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk 
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jelenalazovic/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

Loading raw data

In [ ]:
path = kagglehub.dataset_download("shuyangli94/foodcom-recipes-with-search-terms-and-tags")
csv_path = os.path.join(path, 'recipes_w_search_terms.csv')
df = pd.read_csv(csv_path)

def get_cuisine(x):
    if not isinstance(x, str):
        return None
    x_lower = x.lower()
    if 'italian' in x_lower:
        return 'Italian'
    if 'indian' in x_lower:
        return 'Indian'
    return None

italian_indian_df = df[df['search_terms'].apply(lambda x: isinstance(x, str) and ('italian' in x.lower() or 'indian' in x.lower()))].copy()
italian_indian_df['cuisine'] = italian_indian_df['search_terms'].apply(get_cuisine)
final_df = italian_indian_df[['name', 'steps', 'cuisine']].reset_index(drop=True)
final_df['steps'] = final_df['steps'].apply(lambda x: ' '.join(ast.literal_eval(x)) if isinstance(x, str) else x)
final_df['name'] = final_df['name'].str.lower()
final_df['steps'] = final_df['steps'].str.lower()

indian_all = final_df[final_df['cuisine'] == 'Indian']
italian_sample = final_df[final_df['cuisine'] == 'Italian'].sample(n=len(indian_all), random_state=42)

final_df = pd.concat([italian_sample, indian_all], ignore_index=True)
final_df.to_csv('./data/recipes_raw.csv')

Basic checks

In [31]:
print('Number of null rows\n', final_df.isna().sum())
print('Number of duplicated rows', final_df.duplicated().sum())
df_no_dub = final_df.drop_duplicates().reset_index(drop=True)

Number of null rows
 name       0
steps      0
cuisine    0
dtype: int64
Number of duplicated rows 7


Text cleaning (HTML entities, invisible/control characters)

In [32]:
# zero-width space (200b), line/paragraph separator (2028/2029), BOM (feff), nbsp (a0)
_invisible = [0x200b, 0x2028, 0x2029, 0xfeff, 0xa0]
INVISIBLE_CHARS = re.compile('[' + ''.join(chr(c) for c in _invisible) + ']')
CONTROL_CHARS = re.compile('[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def clean_text(text):
    if not isinstance(text, str):
        return text
    text = html.unescape(text)  # &amp; -> &, &rsquo; -> ’, &eacute; -> é, ...
    text = unicodedata.normalize('NFKC', text)
    text = INVISIBLE_CHARS.sub(' ', text)
    text = CONTROL_CHARS.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean = df_no_dub.copy()
df_clean['name'] = df_clean['name'].apply(clean_text)
df_clean['steps'] = df_clean['steps'].apply(clean_text)

Tokenization and recipe name preparation (name_tokens)

In [33]:
df_clean['name_tokens'] = df_clean['name'].apply(word_tokenize)
df_clean['steps_tokens'] = df_clean['steps'].apply(word_tokenize)
df_clean.to_csv('./data/recipes_tokenized.csv')

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

def remove_punctuation(tokens):
    return [token for token in tokens if token not in string.punctuation]

df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_stopwords)
df_clean['name_tokens'] = df_clean['name_tokens'].apply(remove_punctuation)
# steps_tokens stays untouched here - it's the seq2seq target; stopwords/punctuation
# are removed later from a copy (steps_tokens_bow) for the BoW classifier.

Statistical analysis

In [35]:
df_clean['number_of_tokens'] = df_clean['steps_tokens'].apply(len)
print(df_clean['number_of_tokens'].describe())

max_len = int(df_clean['number_of_tokens'].quantile(0.95))
df_clean_no_outliers = df_clean[
    df_clean['number_of_tokens'].between(13, max_len - 1)
].copy()
print(df_clean_no_outliers['cuisine'].value_counts())

count    13105.000000
mean       143.451507
std         96.299976
min          2.000000
25%         81.000000
50%        122.000000
75%        181.000000
max       1395.000000
Name: number_of_tokens, dtype: float64
cuisine
Indian     6236
Italian    6133
Name: count, dtype: int64


Additional steps_tokens cleaning (numbers, units, artifacts, fractions, hyphens)

In [ ]:
MEASUREMENT_WORDS = {
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    "qt", "quart", "quarts",
    "pint", "pints",
    "g", "kg", "mg",
    "ml", "l",
    "inch", "inches",
    "degree", "degrees",
    "minute", "minutes",
    "hour", "hours"
}

def remove_numbers_measurements(tokens):
    cleaned = []
    for token in tokens:
        token = token.lower()
        # remove pure numbers and fractions
        if re.fullmatch(r"[\d¼½¾⁄/.-]+", token):
            continue
        if token in MEASUREMENT_WORDS:
            continue
        cleaned.append(token)
    return cleaned

ARTIFACTS = {
    "'s", "'re", "'ve", "'ll", "'d", "'m", "n't", "--"
}

def remove_artifacts(tokens):
    return [t for t in tokens if t not in ARTIFACTS]

def normalize_unicode_fractions(tokens):
    replacements = {"½": "1/2", "¼": "1/4", "¾": "3/4", "⁄": "/"}
    normalized = []
    for token in tokens:
        for old, new in replacements.items():
            token = token.replace(old, new)
        normalized.append(token)
    return normalized

def split_hyphenated(tokens):
    output = []
    for token in tokens:
        output.extend(token.replace("-", " ").split())
    return output

# these transforms go into steps_tokens_bow (a copy), not steps_tokens -
# steps_tokens stays raw as the seq2seq target (word2vec vocabulary).
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens"].apply(remove_numbers_measurements)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_artifacts)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(normalize_unicode_fractions)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(split_hyphenated)

Branch: BoW features for the content classifier (steps_tokens_bow)

steps_tokens stays untouched as the seq2seq target. steps_tokens_bow is a copy with stopwords/punctuation removed, used only for the BoW/multi-task classifier.

In [37]:
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_stopwords)
df_clean_no_outliers["steps_tokens_bow"] = df_clean_no_outliers["steps_tokens_bow"].apply(remove_punctuation)

df_clean_no_outliers.to_csv('./data/recipes_final.csv')

Train/val/test split

In [38]:
from sklearn import model_selection

df = pd.read_csv('./data/recipes_final.csv', index_col=0)
df['steps_tokens'] = df['steps_tokens'].apply(ast.literal_eval)
df['steps_tokens_bow'] = df['steps_tokens_bow'].apply(ast.literal_eval)

X = df[['steps_tokens', 'steps_tokens_bow']]
y = df['cuisine']

X_train, X_temp, y_train, y_temp = model_selection.train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = model_selection.train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [39]:
train_df = X_train.join(y_train)
val_df = X_val.join(y_val)
test_df = X_test.join(y_test)

train_df.to_csv('./data/recipes_train.csv')
val_df.to_csv('./data/recipes_val.csv')
test_df.to_csv('./data/recipes_test.csv')

print(train_df.shape, val_df.shape, test_df.shape)

(8658, 3) (1855, 3) (1856, 3)


Word2Vec

In [40]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=train_df['steps_tokens'],
    vector_size=150,
    window=5,
    min_count=2,
    workers=4,
    epochs=15
)

model.wv.save('./data/models/word2vec.wordvectors')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


BoW

In [41]:
from sklearn.feature_extraction.text import CountVectorizer
import joblib

def identity_analyzer(tokens):
    return tokens

vectorizer = CountVectorizer(analyzer=identity_analyzer)

X_train_bow = vectorizer.fit_transform(train_df['steps_tokens_bow'])
X_val_bow = vectorizer.transform(val_df['steps_tokens_bow'])
X_test_bow = vectorizer.transform(test_df['steps_tokens_bow'])

joblib.dump(vectorizer, './data/models/bow_vectorizer.joblib')

['./data/models/bow_vectorizer.joblib']

In [42]:
import numpy as np
from gensim.models import KeyedVectors

wv = KeyedVectors.load('./data/models/word2vec.wordvectors')

seq_len = int(train_df['steps_tokens'].apply(len).quantile(0.75))

def tokens_to_matrix(tokens, wv, seq_len):
    matrix = np.zeros((seq_len, wv.vector_size), dtype=np.float32)
    for i, token in enumerate(tokens[:seq_len]):
        if token in wv:
            matrix[i] = wv[token]
    return matrix

Vocabulary and target indices (for CrossEntropyLoss)

In [ ]:
PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN = '<PAD>', '<UNK>', '<SOS>', '<EOS>'

word2idx = dict(wv.key_to_index)
word2idx[PAD_TOKEN] = len(word2idx)
PAD_IDX = word2idx[PAD_TOKEN]
word2idx[UNK_TOKEN] = len(word2idx)
UNK_IDX = word2idx[UNK_TOKEN]
word2idx[SOS_TOKEN] = len(word2idx)
SOS_IDX = word2idx[SOS_TOKEN]
word2idx[EOS_TOKEN] = len(word2idx)
EOS_IDX = word2idx[EOS_TOKEN]

VOCAB_SIZE = len(word2idx)

def tokens_to_indices(tokens, word2idx, seq_len):
    # leave room for EOS, then pad - EOS is included in the loss (unlike PAD),
    # giving the model a signal for where to stop generating
    tokens = tokens[:seq_len - 1]
    indices = [word2idx.get(t, UNK_IDX) for t in tokens]
    indices.append(EOS_IDX)
    indices += [PAD_IDX] * (seq_len - len(indices))
    return indices

GRU Encoder

In [ ]:
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

class GRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, style_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.content_dim = hidden_dim - style_dim
        self.style_head = nn.Linear(hidden_dim, style_dim)
        self.content_head = nn.Linear(hidden_dim, self.content_dim)

    def forward(self, x):
        _, hidden = self.gru(x)
        hidden = hidden.squeeze(0)                    # (batch, hidden_dim)
        style_latent = self.style_head(hidden)          # (batch, style_dim)
        content_latent = self.content_head(hidden)       # (batch, content_dim)
        return style_latent, content_latent

# encoder turns each recipe into one hidden vector, split via two heads into
# style_latent (e.g. cuisine) and content_latent (recipe content)
input_dim = wv.vector_size
hidden_dim = 512
STYLE_DIM = 64
encoder = GRUEncoder(input_dim, hidden_dim, STYLE_DIM).to(device)

GRU Decoder

In [ ]:
vocab_size = VOCAB_SIZE
MAX_LEN = seq_len

class GRUDecoder(nn.Module):
    def __init__(self, hidden_dim, vocab_size, embed_dim, sos_idx, pad_idx, max_len):
        super().__init__()
        self.max_len = max_len
        self.sos_idx = sos_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru_cell = nn.GRUCell(embed_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, hidden, target=None, teacher_forcing_ratio=0.5):
        # hidden: (batch, hidden_dim) - encoder context vector, used as initial GRU state
        batch_size = hidden.size(0)
        device = hidden.device
        max_len = target.size(1) if target is not None else self.max_len

        input_idx = torch.full((batch_size,), self.sos_idx, dtype=torch.long, device=device)
        h = hidden
        outputs = []

        for t in range(max_len):
            embedded = self.embedding(input_idx)   # (batch, embed_dim)
            h = self.gru_cell(embedded, h)          # (batch, hidden_dim)
            logits_t = self.fc_out(h)               # (batch, vocab_size)
            outputs.append(logits_t.unsqueeze(1))

            use_teacher_forcing = target is not None and torch.rand(1).item() < teacher_forcing_ratio
            input_idx = target[:, t] if use_teacher_forcing else logits_t.argmax(dim=-1)

        return torch.cat(outputs, dim=1)  # (batch, max_len, vocab_size)

decoder = GRUDecoder(hidden_dim, vocab_size, input_dim, SOS_IDX, PAD_IDX, MAX_LEN).to(device)

Seq2Seq Autoencoder — training (input recipe = target recipe)

In [ ]:
from torch.utils.data import Dataset, DataLoader

CUISINE2IDX = {'Italian': 0, 'Indian': 1}
BOW_VOCAB_SIZE = X_train_bow.shape[1]

class RecipeDataset(Dataset):
    def __init__(self, df, wv, word2idx, seq_len, cuisine2idx, bow_matrix):
        self.tokens = df['steps_tokens'].tolist()
        self.cuisines = df['cuisine'].tolist()
        self.wv = wv
        self.word2idx = word2idx
        self.seq_len = seq_len
        self.cuisine2idx = cuisine2idx
        self.bow_matrix = bow_matrix

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        tokens = self.tokens[idx]
        x = tokens_to_matrix(tokens, self.wv, self.seq_len)
        y = tokens_to_indices(tokens, self.word2idx, self.seq_len)
        style_label = self.cuisine2idx[self.cuisines[idx]]
        bow_target = (self.bow_matrix[idx].toarray().ravel() > 0).astype(np.float32)  # multi-label word presence
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(style_label, dtype=torch.long),
            torch.tensor(bow_target, dtype=torch.float32),
        )

BATCH_SIZE = 32

# subsample train_df for faster iteration; 
SAMPLE_SIZE = 2000
train_sample_df = train_df.sample(n=SAMPLE_SIZE, random_state=42)
sample_positions = train_df.index.get_indexer(train_sample_df.index)
train_sample_df = train_sample_df.reset_index(drop=True)
X_train_bow_sample = X_train_bow[sample_positions]

train_dataset = RecipeDataset(train_sample_df, wv, word2idx, seq_len, CUISINE2IDX, X_train_bow_sample)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

class Seq2SeqAutoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, target=None, teacher_forcing_ratio=0.5, return_latents=False):
        style_latent, content_latent = self.encoder(x)
        context = torch.cat([style_latent, content_latent], dim=-1)  # merge back for the decoder
        logits = self.decoder(context, target=target, teacher_forcing_ratio=teacher_forcing_ratio)
        if return_latents:
            return logits, style_latent, content_latent
        return logits

model = Seq2SeqAutoencoder(encoder, decoder).to(device)
style_classifier = nn.Linear(STYLE_DIM, len(CUISINE2IDX)).to(device)

# content classifier: content_vector -> predicted BoW word presence
content_classifier = nn.Sequential(
    nn.Linear(encoder.content_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, BOW_VOCAB_SIZE),
).to(device)

# adversarial style classifier (J_adv(c)): content_latent should carry no style info, so the
# encoder is trained to maximize this classifier's prediction entropy
adv_style_classifier = nn.Sequential(
    nn.Linear(encoder.content_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, len(CUISINE2IDX)),
).to(device)

recon_criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
style_criterion = nn.CrossEntropyLoss()
content_criterion = nn.BCEWithLogitsLoss()  # multi-label: each word independently yes/no
adv_criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(model.parameters()) + list(style_classifier.parameters()) + list(content_classifier.parameters()),
    lr=1e-3,
)
adv_optimizer = torch.optim.Adam(adv_style_classifier.parameters(), lr=1e-3)

NUM_EPOCHS = 100
TEACHER_FORCING_RATIO = 0.5
STYLE_LOSS_WEIGHT = 1.0
CONTENT_LOSS_WEIGHT = 1.0
ADV_LOSS_WEIGHT = 0.5  

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    style_classifier.train()
    content_classifier.train()
    adv_style_classifier.train()
    total_loss = 0.0
    total_recon_loss = 0.0
    total_style_loss = 0.0
    total_content_loss = 0.0
    total_adv_encoder_loss = 0.0
    total_adv_clf_loss = 0.0
    correct_adv = 0
    total_adv = 0

    for batch_x, batch_y, batch_style, batch_bow in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_style = batch_style.to(device)
        batch_bow = batch_bow.to(device)
        optimizer.zero_grad()

        logits, style_latent, content_latent = model(
            batch_x, target=batch_y, teacher_forcing_ratio=TEACHER_FORCING_RATIO, return_latents=True
        )
        recon_loss = recon_criterion(logits.permute(0, 2, 1), batch_y)   # (batch, vocab_size, seq_len) vs (batch, seq_len)

        style_logits = style_classifier(style_latent)
        style_loss = style_criterion(style_logits, batch_style)

        content_logits = content_classifier(content_latent)
        content_loss = content_criterion(content_logits, batch_bow)

        adv_logits_for_encoder = adv_style_classifier(content_latent)
        adv_log_probs = torch.log_softmax(adv_logits_for_encoder, dim=-1)
        adv_entropy = -(adv_log_probs.exp() * adv_log_probs).sum(dim=-1).mean()
        adv_encoder_loss = -adv_entropy

        loss = (
            recon_loss
            + STYLE_LOSS_WEIGHT * style_loss
            + CONTENT_LOSS_WEIGHT * content_loss
            + ADV_LOSS_WEIGHT * adv_encoder_loss
        )
        loss.backward()
        optimizer.step()

        adv_optimizer.zero_grad()
        adv_clf_logits = adv_style_classifier(content_latent.detach())
        adv_clf_loss = adv_criterion(adv_clf_logits, batch_style)
        adv_clf_loss.backward()
        adv_optimizer.step()

        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_style_loss += style_loss.item()
        total_content_loss += content_loss.item()
        total_adv_encoder_loss += adv_encoder_loss.item()
        total_adv_clf_loss += adv_clf_loss.item()
        correct_adv += (adv_clf_logits.argmax(dim=-1) == batch_style).sum().item()
        total_adv += batch_style.size(0)

    n_batches = len(train_loader)
    print(
        f' loss {total_loss / n_batches:.4f} '
        f'| recon {total_recon_loss / n_batches:.4f} | style {total_style_loss / n_batches:.4f} '
        f'| content {total_content_loss / n_batches:.4f} '
        f'| adv_enc {total_adv_encoder_loss / n_batches:.4f} | adv_clf {total_adv_clf_loss / n_batches:.4f} '
        f'| adv_clf_acc {correct_adv / total_adv:.4f}'
    )

In [ ]:
idx2word = {idx: word for word, idx in word2idx.items()}

model.eval()
with torch.no_grad():
    sample_x, sample_y, sample_style, sample_bow = next(iter(train_loader))
    sample_x = sample_x.to(device)
    logits = model(sample_x, target=None, teacher_forcing_ratio=0.0)  # free-running generation, no teacher forcing
    predicted_indices = logits.argmax(dim=-1)  # (batch, seq_len)

def indices_to_tokens(indices):
    # stop at the first EOS - anything after it is not part of the generated content
    tokens = []
    for i in indices:
        if i == EOS_IDX:
            break
        if i != PAD_IDX:
            tokens.append(idx2word[i])
    return tokens

recipe_idx = 0
original_tokens = indices_to_tokens(sample_y[recipe_idx].tolist())
predicted_tokens = indices_to_tokens(predicted_indices[recipe_idx].tolist())

print('ORIGINAL: ', ' '.join(original_tokens))
print('PREDICTED:', ' '.join(predicted_tokens))